# 15. Dask — `dask`
**Python Libraries Lab · Data Science**

_pandas-style code for data bigger than memory, in parallel._

pandas loads the whole table into memory. When the data is bigger than your RAM, that fails. Dask splits the data into many smaller pandas pieces called partitions, and works on them in parallel. The code looks almost exactly like pandas, so you do not have to learn a new style.

**In this notebook:** You will split the ShopEasy year into twelve monthly CSV files, load them all at once with a wildcard, and run the same revenue by city analysis in parallel. Then you will parallelise ordinary Python functions with dask delayed.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
Colab already includes this library. This cell just makes sure it is there (it takes a few seconds).

In [ ]:
!pip install -q "dask[dataframe]"
import importlib.metadata as md
print("dask version:", md.version("dask"))

## Step 1 · Make some monthly files
We create 12 CSV files (one per month), like a real data folder.

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path

rng = np.random.default_rng(42)
Path("monthly").mkdir(exist_ok=True)
for m in range(1, 13):
    n = 20_000
    pd.DataFrame({
        "month": m,
        "city": rng.choice(["Mumbai", "Delhi", "Pune", "Bengaluru"], n),
        "category": rng.choice(["Books", "Electronics", "Clothing", "Home"], n),
        "revenue": rng.integers(199, 9000, n),
    }).to_csv(f"monthly/orders_2026_{m:02d}.csv", index=False)
print(sorted(p.name for p in Path("monthly").glob("*.csv"))[:3], "... 12 files,", 12 * 20_000, "rows")

## Step 2 · Read many files at once
A wildcard reads them all. Nothing is loaded yet, we only see the structure.

In [ ]:
import dask.dataframe as dd

ddf = dd.read_csv("monthly/orders_2026_*.csv")
print("Partitions:", ddf.npartitions)
ddf

## Step 3 · Same code as pandas, then compute()
The groupby builds a plan. `.compute()` runs it in parallel and returns a pandas result.

In [ ]:
plan = ddf.groupby("city")["revenue"].sum()
print(type(plan))                 # still lazy
result = plan.compute()
print(type(result))
result.sort_values(ascending=False)

## Step 4 · Several results in one pass
`dask.compute` shares the work between results instead of reading the files twice.

In [ ]:
import dask
total, by_month = dask.compute(ddf["revenue"].sum(), ddf.groupby("month")["revenue"].mean())
print(f"Year revenue: Rs.{total:,}")
by_month.round(0).sort_index().head()

## Step 5 · dask.delayed for ordinary functions
Wrap normal functions so they run in parallel. Here 6 one-second jobs take about 1-2 seconds.

In [ ]:
import time
from dask import delayed

def slow_report(city):
    time.sleep(1)
    return f"report for {city} ready"

cities = ["Mumbai", "Delhi", "Pune", "Bengaluru", "Chennai", "Jaipur"]
jobs = [delayed(slow_report)(c) for c in cities]
t = time.perf_counter()
out = dask.compute(*jobs, scheduler="threads")
print(out)
print(f"Took {time.perf_counter() - t:.1f}s instead of ~6s")

## ✍️ Your turn
Using `ddf`, store the **number of rows where category is 'Books'** in `answer`. Remember `.compute()`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    expected = int(sum((pd.read_csv(p)["category"] == "Books").sum() for p in Path("monthly").glob("*.csv")))
    return answer == expected, f"Expected {expected}, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = int((ddf["category"] == "Books").sum().compute())

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Partition**
- **Lazy evaluation**
- **Task graph**
- **.compute()**
- **dask.delayed**

**Next:** 16. statsmodels (`statsmodels`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/16_statsmodels.ipynb)

## 💻 Run it on your own computer (optional)
1. Install Python 3.10+ from python.org (on Windows tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install dask[dataframe]
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that just opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain `.py` file, run `!` commands in the terminal instead.